# Daily Solar Yield Inference

Loads a trained MLflow model, scores the latest Open-Meteo forecast, applies physical guardrails, and projects the result into plane-of-array GTI.


In [0]:
%pip install openmeteo-requests pvlib pandas pyarrow xgboost scikit-learn seaborn requests
dbutils.library.restartPython()

In [ ]:
from pathlib import Path
import sys

import mlflow.sklearn
import openmeteo_requests
import pandas as pd


def add_repo_src_to_path():
    candidates = []
    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])

    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        workspace_path = Path("/Workspace") / notebook_path.lstrip("/")
        workspace_dir = workspace_path.parent
        candidates.extend([workspace_dir, *workspace_dir.parents])
    except Exception:
        pass

    for candidate in candidates:
        src_path = candidate / "src"
        if (src_path / "solar_yield" / "features.py").exists():
            src_path_text = str(src_path)
            if src_path_text not in sys.path:
                sys.path.insert(0, src_path_text)
            return candidate

    raise RuntimeError("Could not locate src/solar_yield/features.py from this notebook.")


repo_root = add_repo_src_to_path()
print(f"Using shared package source from: {repo_root / 'src'}")

from solar_yield.features import (  # noqa: E402
    FEATURE_COLUMNS,
    add_model_features,
    apply_physical_overrides,
    prepare_model_matrix,
)
from solar_yield.quality import validate_hourly_forecast  # noqa: E402

# ==========================================
# 1. ARCHITECTURE SETUP & CONFIGURATION
# ==========================================
dbutils.widgets.text("model_uri", "", "Exact MLflow model URI")
dbutils.widgets.text("model_run_id", "", "MLflow model run ID fallback")
MODEL_URI = dbutils.widgets.get("model_uri").strip()
RUN_ID = dbutils.widgets.get("model_run_id").strip()
if not MODEL_URI:
    if not RUN_ID:
        raise ValueError(
            "Set model_uri from the training notebook output, or set model_run_id "
            "for fallback runs:/ loading."
        )
    MODEL_URI = f"runs:/{RUN_ID}/solar_factor_model"

print(f"Loading production time-aware weighted chained model from MLflow: {MODEL_URI}")
model = mlflow.sklearn.load_model(MODEL_URI)

# ==========================================
# 2. BRONZE LAYER: LIVE FORECAST INGESTION
# ==========================================
om = openmeteo_requests.Client()
FORECAST_TIMEZONE = "Australia/Perth"
MODEL_TIMEZONE = "UTC"
spark.conf.set("spark.sql.session.timeZone", MODEL_TIMEZONE)

params = {
    "latitude": -31.95,
    "longitude": 115.86,
    "hourly": [
        "cloud_cover_low",
        "cloud_cover_mid",
        "cloud_cover_high",
        "total_column_integrated_water_vapour",
        "sunshine_duration",
        "temperature_2m",
        "relative_humidity_2m",
        "surface_pressure",
    ],
    "timezone": FORECAST_TIMEZONE,
    "forecast_days": 7,
}
responses = om.weather_api("https://api.open-meteo.com/v1/forecast", params=params)
hourly = responses[0].Hourly()

start_epoch = hourly.Time()
end_epoch = hourly.TimeEnd()
step_seconds = hourly.Interval()

# Keep timestamps in UTC for model features and pvlib so inference matches training.
date_range = pd.date_range(
    start=pd.to_datetime(start_epoch, unit="s", utc=True),
    end=pd.to_datetime(end_epoch, unit="s", utc=True),
    freq=pd.Timedelta(seconds=step_seconds),
    inclusive="left",
)

print(f"Generated timestamp vector length: {len(date_range)}")
print(f"Generated weather feature length: {len(hourly.Variables(0).ValuesAsNumpy())}")

pdf_forecast_raw = pd.DataFrame(
    {
        "timestamp": date_range,
        "cloud_low": hourly.Variables(0).ValuesAsNumpy(),
        "cloud_mid": hourly.Variables(1).ValuesAsNumpy(),
        "cloud_high": hourly.Variables(2).ValuesAsNumpy(),
        "water_vapour": hourly.Variables(3).ValuesAsNumpy(),
        "sunshine_duration": hourly.Variables(4).ValuesAsNumpy(),
        "temperature": hourly.Variables(5).ValuesAsNumpy(),
        "relative_humidity": hourly.Variables(6).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(7).ValuesAsNumpy(),
    }
)

validate_hourly_forecast(pdf_forecast_raw, expected_rows=168)

# Keep a Bronze Spark frame for Databricks layer visibility and optional inspection.
df_forecast_bronze = spark.createDataFrame(pdf_forecast_raw)

# ==========================================
# 3. SILVER LAYER: SHARED TIME-AWARE FEATURES
# ==========================================
print("Engineering shared time-aware inference features...")
X_inference_full = add_model_features(pdf_forecast_raw, mode="inference")
X_features = prepare_model_matrix(X_inference_full, FEATURE_COLUMNS)

if len(X_features) != 168:
    raise ValueError(f"expected 168 inference feature rows, got {len(X_features)}")
if X_features.isna().any().any():
    raise ValueError("inference feature matrix contains null values")

print(f"Inference feature matrix shape: {X_features.shape}")


def first_estimator_feature_names(loaded_model):
    estimators = getattr(loaded_model, "estimators_", [])
    if not estimators:
        return None
    estimator = estimators[0]
    feature_names = getattr(estimator, "feature_names_in_", None)
    if feature_names is not None:
        return list(feature_names)
    booster_getter = getattr(estimator, "get_booster", None)
    if booster_getter is not None:
        booster = booster_getter()
        if getattr(booster, "feature_names", None):
            return list(booster.feature_names)
    return None


expected_feature_names = first_estimator_feature_names(model)
if expected_feature_names is not None:
    missing = [column for column in expected_feature_names if column not in X_features.columns]
    unexpected = [column for column in X_features.columns if column not in expected_feature_names]
    print(f"Loaded model first-estimator feature count: {len(expected_feature_names)}")
    if missing or unexpected:
        raise ValueError(
            "Loaded model feature schema does not match the inference feature matrix. "
            "Use the exact model_uri printed by the Phase 3 training notebook. "
            f"Missing columns: {missing[:10]}; unexpected columns: {unexpected[:10]}"
        )

# Keep a Silver Spark frame for Databricks layer visibility and optional inspection.
df_forecast_silver = spark.createDataFrame(X_inference_full)

# ==========================================
# 4. GOLD LAYER: INFERENCE & PHYSICAL OVERRIDES
# ==========================================
print("Generating predictions and applying physical overrides...")
raw_predictions = model.predict(X_features)
X_inference_full = apply_physical_overrides(X_inference_full, raw_predictions)

# Convert finalized arrays to Spark Gold Layer for production storage/BI dashboarding.
df_forecast_gold = spark.createDataFrame(
    X_inference_full[
        [
            "timestamp",
            "temperature",
            "sunshine_fraction",
            "final_pred_direct",
            "final_pred_diffuse",
        ]
    ]
)

# Write out to your production Delta lake directory.
# df_forecast_gold.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.default.solar_power_7d_forecast")

print("==================================================================")
print("SUCCESS: 7-Day Production Forecast Successfully Written to Gold!")
print(f"Generated {df_forecast_gold.count()} rows of clear sky solar attenuation vectors.")
print("==================================================================")


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

FORECAST_TIMEZONE = "Australia/Perth"
FIGURE_DIR = Path("/Volumes/main/default/solar-figures")
ATTENUATION_IMAGE_NAME = "7_Day_Attenuation_Forecast_Profile.png"
GTI_IMAGE_NAME = "7_Day_GTI_Power_Yield_Profile.png"
attenuation_save_path = str(FIGURE_DIR / ATTENUATION_IMAGE_NAME)
gti_save_path = str(FIGURE_DIR / GTI_IMAGE_NAME)

FORECAST_COLORS = {
    "direct": "#2563eb",
    "diffuse": "#f97316",
    "sunshine": "#16a34a",
    "gti_total": "#dc2626",
    "gti_direct": "#f59e0b",
    "gti_diffuse": "#0284c7",
}


def local_timestamp_series(series):
    timestamps = pd.to_datetime(series)
    if timestamps.dt.tz is None:
        timestamps = timestamps.dt.tz_localize(MODEL_TIMEZONE)
    else:
        timestamps = timestamps.dt.tz_convert(MODEL_TIMEZONE)
    return timestamps.dt.tz_convert(FORECAST_TIMEZONE).dt.tz_localize(None)


def apply_forecast_time_axis(ax, timestamps, show_xlabel=False):
    ax.set_xlim(timestamps.min(), timestamps.max())
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%H:%M"))
    ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))
    ax.grid(True, which="major", axis="both", color="#d0d7de", linewidth=0.9)
    ax.grid(True, which="minor", axis="x", color="#e5e7eb", linewidth=0.6, alpha=0.7)
    ax.tick_params(axis="both", labelsize=11)
    if show_xlabel:
        ax.set_xlabel("7-Day Forecast Window (Perth Local Time)", fontsize=13, labelpad=12)


FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update(
    {
        "axes.titlesize": 17,
        "axes.labelsize": 13,
        "legend.fontsize": 11,
        "figure.dpi": 130,
        "savefig.dpi": 170,
    }
)

pdf_fixed = df_forecast_gold.orderBy("timestamp").toPandas()
pdf_fixed["timestamp"] = local_timestamp_series(pdf_fixed["timestamp"])
pdf_fixed = pdf_fixed.sort_values("timestamp").reset_index(drop=True)
time_axis = pdf_fixed["timestamp"]

fig, axs = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle("7-Day Solar Attenuation Forecast", fontsize=20, fontweight="bold", y=0.98)

axs[0].plot(
    time_axis,
    pdf_fixed["final_pred_direct"],
    color=FORECAST_COLORS["direct"],
    linewidth=2.4,
    label="Direct attenuation",
)
axs[0].fill_between(
    time_axis,
    pdf_fixed["final_pred_direct"],
    color=FORECAST_COLORS["direct"],
    alpha=0.12,
)
axs[0].set_ylabel("Direct")
axs[0].set_ylim(-0.05, 1.05)
axs[0].legend(loc="upper right", frameon=True)

axs[1].plot(
    time_axis,
    pdf_fixed["final_pred_diffuse"],
    color=FORECAST_COLORS["diffuse"],
    linewidth=2.4,
    label="Diffuse attenuation",
)
axs[1].fill_between(
    time_axis,
    pdf_fixed["final_pred_diffuse"],
    color=FORECAST_COLORS["diffuse"],
    alpha=0.12,
)
axs[1].set_ylabel("Diffuse")
axs[1].set_ylim(-0.05, 1.05)
axs[1].legend(loc="upper right", frameon=True)

axs[2].plot(
    time_axis,
    pdf_fixed["sunshine_fraction"],
    color=FORECAST_COLORS["sunshine"],
    linewidth=2.0,
    linestyle="--",
    label="Sunshine fraction",
)
axs[2].fill_between(
    time_axis,
    pdf_fixed["sunshine_fraction"],
    color=FORECAST_COLORS["sunshine"],
    alpha=0.08,
)
axs[2].set_ylabel("Sunshine")
axs[2].set_ylim(-0.05, 1.05)
axs[2].legend(loc="upper right", frameon=True)

for ax in axs:
    apply_forecast_time_axis(ax, time_axis)
apply_forecast_time_axis(axs[-1], time_axis, show_xlabel=True)

fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(attenuation_save_path, bbox_inches="tight")
print(f"Attenuation forecast figure saved to: {attenuation_save_path}")
plt.show()


In [0]:
import pvlib
from pvlib.location import Location
import pandas as pd
import numpy as np

# 1. Clear Site Geometry
LATITUDE = -31.95
LONGITUDE = 115.86
TZ = "Australia/Perth"
SURFACE_TILT = 25.0     # Ideal tilt for Perth
SURFACE_AZIMUTH = 0.0   # 0 = True North (Southern Hemisphere)
ALBEDO = 0.2           

print("Initializing localized geometry engine...")
site = Location(latitude=LATITUDE, longitude=LONGITUDE, tz=TZ)

# 2. Pull data from Spark and enforce strict Localization
df_gold_predictions = df_forecast_gold.orderBy("timestamp").toPandas()

# Interpret inference timestamps as UTC instants; pvlib computes solar position for the Perth site.
df_gold_predictions["timestamp"] = pd.to_datetime(df_gold_predictions["timestamp"])
if df_gold_predictions["timestamp"].dt.tz is None:
    df_gold_predictions.index = df_gold_predictions["timestamp"].dt.tz_localize(MODEL_TIMEZONE)
else:
    df_gold_predictions.index = df_gold_predictions["timestamp"].dt.tz_convert(MODEL_TIMEZONE)

# 3. Compute precise solar track vectors matching local hours
solar_position = site.get_solarposition(df_gold_predictions.index)
zenith = solar_position['zenith']
apparent_elevation = solar_position['apparent_elevation']
azimuth = solar_position['azimuth']

# 4. Generate the localized clear sky envelopes
clear_sky = site.get_clearsky(df_gold_predictions.index)

# 5. Apply your model's prediction fractions
attenuated_dni = clear_sky['dni'] * df_gold_predictions['final_pred_direct']
attenuated_dhi = clear_sky['dhi'] * df_gold_predictions['final_pred_diffuse']
attenuated_ghi = (attenuated_dni * np.cos(np.radians(zenith))) + attenuated_dhi

# 6. Total Plane-of-Array (POA) GTI calculation
print("Projecting paths onto North-facing planes...")
total_gti = pvlib.irradiance.get_total_irradiance(
    surface_tilt=SURFACE_TILT,
    surface_azimuth=SURFACE_AZIMUTH,
    solar_zenith=zenith,
    solar_azimuth=azimuth,
    dni=attenuated_dni,
    ghi=attenuated_ghi,
    dhi=attenuated_dhi,
    albedo=ALBEDO,
    model='isotropic'
)

# 7. Map variables back to your clean dataframe
df_gold_predictions["gti_total"] = total_gti['poa_global'].values
df_gold_predictions["gti_direct"] = total_gti['poa_direct'].values
df_gold_predictions["gti_diffuse"] = total_gti['poa_diffuse'].values

# Overwrite Gold Spark table with mathematically corrected profiles
df_final_power_forecast = spark.createDataFrame(df_gold_predictions[[
    "timestamp", "temperature", "sunshine_fraction",
    "final_pred_direct", "final_pred_diffuse",
    "gti_total", "gti_direct", "gti_diffuse"
]])

print("=" * 65)
print("SUCCESS: LOCALIZED GEOMETRIC CORRECTION COMPLETE!")
print(f"Peak predicted 7-day GTI array yield: {df_gold_predictions['gti_total'].max():.2f} W/m²")
print("=" * 65)


In [ ]:
# Diagnostic table for timestamp alignment and GTI peak drivers.
# Run this after the GTI projection cell if the plotted daily shape looks suspicious.
gti_debug = df_gold_predictions.reset_index(drop=True).copy()
gti_debug["timestamp"] = local_timestamp_series(gti_debug["timestamp"])
gti_debug["solar_zenith"] = zenith.to_numpy()
gti_debug["solar_azimuth"] = azimuth.to_numpy()
gti_debug["clear_dni"] = clear_sky["dni"].to_numpy()
gti_debug["clear_dhi"] = clear_sky["dhi"].to_numpy()
gti_debug["gti_total"] = df_gold_predictions["gti_total"].to_numpy()
gti_debug["gti_direct"] = df_gold_predictions["gti_direct"].to_numpy()
gti_debug["gti_diffuse"] = df_gold_predictions["gti_diffuse"].to_numpy()

peak_ts = gti_debug.loc[gti_debug["gti_total"].idxmax(), "timestamp"]
peak_day = peak_ts.date()
gti_peak_day_debug = gti_debug[gti_debug["timestamp"].dt.date == peak_day][
    [
        "timestamp",
        "sunshine_fraction",
        "final_pred_direct",
        "final_pred_diffuse",
        "clear_dni",
        "clear_dhi",
        "solar_zenith",
        "solar_azimuth",
        "gti_total",
        "gti_direct",
        "gti_diffuse",
    ]
].copy()

midday_alignment_check = gti_debug[
    (gti_debug["timestamp"].dt.hour.between(10, 15))
    & (gti_debug["clear_dni"] > 0)
    & (gti_debug["sunshine_fraction"] == 0)
][["timestamp", "sunshine_fraction", "clear_dni", "solar_zenith", "gti_total"]]

print(f"Forecast timestamp range: {gti_debug['timestamp'].min()} -> {gti_debug['timestamp'].max()}")
print(f"Peak GTI timestamp: {peak_ts}")
print(f"Peak-day diagnostic rows: {len(gti_peak_day_debug)}")
print(f"Midday clear-sky rows with zero sunshine_fraction: {len(midday_alignment_check)}")

if len(midday_alignment_check) > 0:
    print("WARNING: sunshine_fraction is zero during clear-sky daylight rows; check timestamp alignment.")
    display(midday_alignment_check.head(12))

display(gti_peak_day_debug)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pdf_yield = df_final_power_forecast.orderBy("timestamp").toPandas()
pdf_yield["timestamp"] = local_timestamp_series(pdf_yield["timestamp"])
pdf_yield = pdf_yield.sort_values("timestamp").reset_index(drop=True)
time_axis = pdf_yield["timestamp"]

sns.set_theme(style="whitegrid", context="talk")
fig, axs = plt.subplots(
    2,
    1,
    figsize=(16, 10),
    sharex=True,
    gridspec_kw={"height_ratios": [2.2, 1.0]},
)
fig.suptitle("7-Day Global Tilted Irradiance Forecast", fontsize=20, fontweight="bold", y=0.98)

axs[0].plot(
    time_axis,
    pdf_yield["gti_total"],
    color=FORECAST_COLORS["gti_total"],
    linewidth=2.8,
    label="Total tilted GTI",
)
axs[0].fill_between(
    time_axis,
    pdf_yield["gti_total"],
    color=FORECAST_COLORS["gti_total"],
    alpha=0.10,
)
axs[0].plot(
    time_axis,
    pdf_yield["gti_direct"],
    color=FORECAST_COLORS["gti_direct"],
    linewidth=2.0,
    linestyle="-.",
    label="Direct POA component",
)
axs[0].plot(
    time_axis,
    pdf_yield["gti_diffuse"],
    color=FORECAST_COLORS["gti_diffuse"],
    linewidth=2.0,
    linestyle="--",
    label="Diffuse POA component",
)
axs[0].set_ylabel("Irradiance (W/m²)")
axs[0].set_ylim(-20, max(50, pdf_yield["gti_total"].max() * 1.10))
axs[0].legend(loc="upper right", frameon=True)

axs[1].plot(
    time_axis,
    pdf_yield["final_pred_direct"],
    color=FORECAST_COLORS["direct"],
    linewidth=2.0,
    label="Direct attenuation",
)
axs[1].plot(
    time_axis,
    pdf_yield["final_pred_diffuse"],
    color=FORECAST_COLORS["diffuse"],
    linewidth=2.0,
    label="Diffuse attenuation",
)
axs[1].plot(
    time_axis,
    pdf_yield["sunshine_fraction"],
    color=FORECAST_COLORS["sunshine"],
    linewidth=1.8,
    linestyle=":",
    label="Sunshine fraction",
)
axs[1].set_ylabel("Factor")
axs[1].set_ylim(-0.05, 1.05)
axs[1].legend(loc="upper right", ncol=3, frameon=True)

for ax in axs:
    apply_forecast_time_axis(ax, time_axis)
apply_forecast_time_axis(axs[-1], time_axis, show_xlabel=True)

fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(gti_save_path, bbox_inches="tight")
save_path = gti_save_path
print(f"GTI forecast figure saved to: {gti_save_path}")
plt.show()


In [0]:
import base64
from datetime import UTC, datetime

import requests

# Diagnostic write probe for Databricks Free Edition + GitHub API publishing.
# Token is stored in Databricks Secret Scope and is never printed.
GITHUB_TOKEN = dbutils.secrets.get(scope="github", key="GITHUB_TOKEN").strip()
GITHUB_REPO = "jun01ee/solar-yield-forecasting-pipeline"
GITHUB_BRANCH = "forecast-artifacts"
PROBE_PATH = "publish_probe.txt"
API_ROOT = "https://api.github.com"

GITHUB_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}


def raise_github_error(action, response):
    raise RuntimeError(
        f"{action} failed. status={response.status_code}, body={response.text}"
    )


def get_existing_file_sha(remote_path):
    url = f"{API_ROOT}/repos/{GITHUB_REPO}/contents/{remote_path}"
    response = requests.get(
        url,
        headers=GITHUB_HEADERS,
        params={"ref": GITHUB_BRANCH},
        timeout=30,
    )
    if response.status_code == 200:
        return response.json()["sha"]
    if response.status_code == 404:
        return None
    raise_github_error(f"Read GitHub file metadata for {remote_path}", response)


def put_github_file(remote_path, content_bytes, message):
    url = f"{API_ROOT}/repos/{GITHUB_REPO}/contents/{remote_path}"
    sha = get_existing_file_sha(remote_path)
    payload = {
        "message": message,
        "content": base64.b64encode(content_bytes).decode("utf-8"),
        "branch": GITHUB_BRANCH,
    }
    if sha:
        payload["sha"] = sha

    response = requests.put(url, headers=GITHUB_HEADERS, json=payload, timeout=60)
    if response.status_code not in (200, 201):
        raise_github_error(f"Write GitHub file {remote_path}", response)
    return response



repo_response = requests.get(
    f"{API_ROOT}/repos/{GITHUB_REPO}",
    headers=GITHUB_HEADERS,
    timeout=30,
)
if repo_response.status_code != 200:
    raise_github_error("Check GitHub repository access", repo_response)

permissions = repo_response.json().get("permissions", {})
print(f"GitHub repository access OK. Reported permissions: {permissions}")
if permissions and not permissions.get("push", False):
    raise RuntimeError(
        "GitHub token can read the repository but does not report push=True. "
        "Use a token with Contents: Read and write for this repository."
    )

branch_response = requests.get(
    f"{API_ROOT}/repos/{GITHUB_REPO}/branches/{GITHUB_BRANCH}",
    headers=GITHUB_HEADERS,
    timeout=30,
)
if branch_response.status_code != 200:
    raise_github_error(f"Check GitHub branch {GITHUB_BRANCH}", branch_response)
print(f"GitHub branch access OK: {GITHUB_BRANCH}")

probe_text = (
    "Databricks GitHub publish probe succeeded at "
    f"{datetime.now(UTC).isoformat()}\n"
)
put_github_file(
    PROBE_PATH,
    probe_text.encode("utf-8"),
    f"Update Databricks publish probe {datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S UTC')}",
)
print(f"GitHub write probe OK: {GITHUB_REPO}/{PROBE_PATH} on {GITHUB_BRANCH}")


In [ ]:
import hashlib
import os
import shutil
import stat
import subprocess
from pathlib import Path

# Production publish: update README forecast images on the artifact branch.
# This uses git CLI because large base64 JSON requests can fail with opaque
# 400 responses in restricted Databricks runtimes.
FORECAST_ARTIFACTS = {
    ATTENUATION_IMAGE_NAME: Path(attenuation_save_path),
    GTI_IMAGE_NAME: Path(gti_save_path),
}
WORKDIR = Path("/tmp/solar_forecast_artifacts_repo")
ASKPASS_PATH = Path("/tmp/github_askpass_forecast.sh")

for remote_path, local_file in FORECAST_ARTIFACTS.items():
    if not local_file.exists():
        raise FileNotFoundError(f"Forecast image was not created: {local_file}")
    image_bytes = local_file.read_bytes()
    print(f"{remote_path} size: {len(image_bytes)} bytes")
    print(f"{remote_path} sha256: {hashlib.sha256(image_bytes).hexdigest()}")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

ASKPASS_PATH.write_text(
    "#!/bin/sh\n"
    "case \"$1\" in\n"
    "  *Username*) echo x-access-token ;;\n"
    "  *Password*) printf '%s\n' \"$GITHUB_TOKEN\" ;;\n"
    "  *) echo ;;\n"
    "esac\n"
)
ASKPASS_PATH.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

git_env = os.environ.copy()
git_env["GIT_ASKPASS"] = str(ASKPASS_PATH)
git_env["GIT_TERMINAL_PROMPT"] = "0"
git_env["GITHUB_TOKEN"] = GITHUB_TOKEN

clone_url = f"https://github.com/{GITHUB_REPO}.git"
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, clone_url, str(WORKDIR)],
    check=True,
    env=git_env,
)

for remote_path, local_file in FORECAST_ARTIFACTS.items():
    shutil.copyfile(local_file, WORKDIR / remote_path)

subprocess.run(["git", "config", "user.name", "Databricks Solar Forecast Bot"], cwd=WORKDIR, check=True)
subprocess.run(["git", "config", "user.email", "actions@users.noreply.github.com"], cwd=WORKDIR, check=True)
subprocess.run(["git", "add", *FORECAST_ARTIFACTS.keys()], cwd=WORKDIR, check=True)

commit_result = subprocess.run(
    [
        "git",
        "commit",
        "-m",
        f"Update daily solar forecast plots {datetime.now(UTC).strftime('%Y-%m-%d')}",
    ],
    cwd=WORKDIR,
    text=True,
    capture_output=True,
)

if commit_result.returncode == 0:
    subprocess.run(["git", "push", "origin", GITHUB_BRANCH], cwd=WORKDIR, check=True, env=git_env)
    print(
        f"Published latest forecast images to {GITHUB_REPO} on {GITHUB_BRANCH}: "
        f"{', '.join(FORECAST_ARTIFACTS.keys())}"
    )
elif "nothing to commit" in (commit_result.stdout + commit_result.stderr).lower():
    print("Forecast images are unchanged; no GitHub commit needed.")
else:
    raise RuntimeError(
        "Git commit for forecast images failed. "
        f"stdout={commit_result.stdout}, stderr={commit_result.stderr}"
    )

try:
    ASKPASS_PATH.unlink()
except FileNotFoundError:
    pass
